# Medical Question Type Classification
## End-to-End Pipeline: BERT + LoRA-Free Fine-Tuning

---

| Component | Detail |
|---|---|
| **Model** | `bert-base-cased` (BERT) |
| **Task** | 4-class classification: Diagnostic · Factoid · Preventive · Prognostic |
| **Split** | Stratified 80 / 10 / 10 (train / val / test) |
| **Imbalance** | Oversampling (all classes) + synonym augmentation (Factoid only) |
| **Evaluation** | Accuracy, Classification Report, Confusion Matrix |

##Steps

```
1  Install & Setup          2  Seeds & GPU
3  Data Loading             4  Text Preprocessing
5  EDA                      6  Stratified Split (80/10/10)
7  Imbalance Handling       8  Tokenisation
9  Dataset & DataLoaders    10 Load BERT
11 Training (Trainer API)   12 Evaluation on Test Set
13 Confusion Matrix         14 Training Curves
15 Sample Predictions       16 Predict Function
```


## 1 - Install Libraries

In [ ]:
# All required packages for the complete pipeline
!pip install -q transformers>=4.40.0 accelerate>=0.30.0
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn tqdm nltk

import nltk
# WordNet for synonym-replacement augmentation
for resource in ['wordnet', 'averaged_perceptron_tagger', 'punkt', 'omw-1.4', 'punkt_tab', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

print('All packages installed.')


## 2 - Seeds, GPU Check & Global Imports

In [ ]:
import os, sys, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight

# ── Fix all random seeds ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# ── GPU ──────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Seed     : {SEED}')


## 3 - Global Configuration & Data Loading

In [ ]:
# ── Global configuration (single source of truth) ────────────────────────────
MODEL_NAME   = 'bert-base-uncased'  # BioBERT — MANDATORY
MAX_LEN      = 128      # BioBERT max token length
BATCH_SIZE   = 8       # Trainer API batch size
EPOCHS       = 2        # Training epochs (Trainer)
LR           = 2e-5     # Learning rate
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# Label mapping
LABEL2ID    = {'Diagnostic': 0, 'Factoid': 1, 'Preventive': 2, 'Prognostic': 3}
ID2LABEL    = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = len(LABEL2ID)
CLASS_NAMES = list(LABEL2ID.keys())
MINORITY_CLS = 'Factoid'   # target of synonym augmentation

# Augmentation cap
AUG_CAP = 300

# Colour palette for plots
PALETTE = {
    'Diagnostic': '#e74c3c',
    'Factoid':    '#3498db',
    'Preventive': '#2ecc71',
    'Prognostic': '#9b59b6',
}

print(f'Model    : {MODEL_NAME}')
print(f'Classes  : {CLASS_NAMES}')

In [ ]:
# ── Upload dataset CSV files ──────────────────────────────────────────────────
# The original notebook concatenates mhqa-b.csv (train) and mhqa.csv (test).
# Here we pool both files and re-split stratified 80/10/10.
from google.colab import files
print('Upload your CSV file(s) containing columns: question, type')
print('Expected: mhqa-b.csv and/or mhqa.csv')
uploaded = files.upload()
for fname, data in uploaded.items():
    print(f'  Uploaded: {fname}  ({len(data):,} bytes)')

In [ ]:
# ── Load and pool all uploaded CSVs ──────────────────────────────────────────
frames = []
for fname in uploaded.keys():
    df_raw = pd.read_csv(fname)
    if not {'question', 'type'}.issubset(df_raw.columns):
        raise ValueError(f'{fname} must contain columns: question, type')
    frames.append(df_raw[['question', 'type']].copy())
    print(f'{fname}: {len(df_raw):,} rows')

df_all = pd.concat(frames, ignore_index=True)
print(f'\nTotal pooled rows: {len(df_all):,}')
print(df_all['type'].value_counts().to_string())


## 4 - Text Preprocessing
**Cleans question text while preserving medical meaning.**

In [ ]:
import re

def preprocess_text(text: str) -> str:
    '''
    Medical-safe text cleaning pipeline:
      1. Strip HTML tags
      2. Remove URLs (http/https)
      3. Collapse repeated whitespace
      4. Lowercase
      5. Remove standalone punctuation sequences
      6. Strip leading/trailing whitespace

    INTENTIONALLY PRESERVED:
      - Medical acronyms: HIV, CT, MRI, CBC, HbA1c
      - Alphanumeric codes: COVID-19, ICD-10
      - Numeric values embedded in words
    '''
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)         # remove HTML tags
    text = re.sub(r'https?://\S+', ' ', text)    # remove URLs
    text = re.sub(r'\s+', ' ', text)             # collapse whitespace
    text = text.lower()
    text = re.sub(r'[^\w\s\-\'.]', ' ', text)  # keep alphanum, hyphen, dot
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def apply_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    '''Apply full preprocessing + quality filters to a DataFrame.'''
    df = df.copy()
    # Clean text
    df['clean_question'] = df['question'].apply(preprocess_text)
    # Drop nulls
    df.dropna(subset=['question', 'type'], inplace=True)
    # Keep valid labels only
    df = df[df['type'].isin(LABEL2ID.keys())].copy()
    # Remove exact duplicates
    df.drop_duplicates(subset=['clean_question'], inplace=True)
    # Remove very short questions (likely noise)
    df = df[df['clean_question'].str.len() >= 15].copy()
    # Encode label
    df['label'] = df['type'].map(LABEL2ID)
    df.reset_index(drop=True, inplace=True)
    return df


df_clean = apply_preprocessing(df_all)
print(f'After preprocessing: {len(df_clean):,} samples')
print(f'\nClass distribution:')
print(df_clean['type'].value_counts().to_string())

# Preview
print('\nPreprocessing sample:')
for _, row in df_clean.head(2).iterrows():
    print(f'  Raw  : {row["question"][:90]}')
    print(f'  Clean: {row["clean_question"][:90]}')
    print()


## 5 - Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('MHQA Dataset — EDA', fontsize=14, fontweight='bold')

# (1) Class distribution
counts = df_clean['type'].value_counts()
cols   = [PALETTE[t] for t in counts.index]
bars   = axes[0].bar(counts.index, counts.values, color=cols,
                     edgecolor='black', linewidth=0.5)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
for b, v in zip(bars, counts.values):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.5, str(v),
                 ha='center', fontsize=10, fontweight='bold')

# (2) Question length (chars)
df_clean['q_len'] = df_clean['clean_question'].str.len()
for cls, grp in df_clean.groupby('type'):
    axes[1].hist(grp['q_len'], bins=25, alpha=0.55,
                 label=cls, color=PALETTE[cls])
axes[1].set_title('Question Length Distribution', fontweight='bold')
axes[1].set_xlabel('Characters')
axes[1].set_ylabel('Frequency')
axes[1].legend(fontsize=8)

# (3) Imbalance ratio table
ir   = counts.max() / counts.min()
axes[2].axis('off')
axes[2].text(0.5, 0.6, f'Imbalance Ratio\n{ir:.2f}x',
             ha='center', va='center', fontsize=22, fontweight='bold',
             color='#e74c3c', transform=axes[2].transAxes)
axes[2].text(0.5, 0.25,
             f'Majority: {counts.index[0]} ({counts.max()})\n'
             f'Minority: {counts.index[-1]} ({counts.min()})',
             ha='center', va='center', fontsize=11,
             transform=axes[2].transAxes,
             bbox=dict(boxstyle='round,pad=0.5',
                       facecolor='#fff3cd', edgecolor='#f0ad4e'))
axes[2].set_title('Imbalance Summary', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Imbalance ratio: {ir:.2f}x  -> correction required in §7')


## 6 - Stratified Train / Validation / Test Split (80 / 10 / 10)
**Stratification preserves class proportions in all three splits.**

In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: 80% train, 20% temp (temp will become val + test)
df_train_raw, df_temp = train_test_split(
    df_clean,
    test_size    = 0.20,
    stratify     = df_clean['label'],
    random_state = SEED
)

# Step 2: split temp 50/50 → val 10%, test 10%
df_val, df_test = train_test_split(
    df_temp,
    test_size    = 0.50,
    stratify     = df_temp['label'],
    random_state = SEED
)

# Reset indices
for df_ in [df_train_raw, df_val, df_test]:
    df_.reset_index(drop=True, inplace=True)

print(f'Train (raw) : {len(df_train_raw):,}')
print(f'Validation  : {len(df_val):,}')
print(f'Test        : {len(df_test):,}')
print(f'Total       : {len(df_train_raw)+len(df_val)+len(df_test):,}')

print('\nTrain class distribution (before balancing):')
print(df_train_raw['type'].value_counts().to_string())

print('\nValidation class distribution:')
print(df_val['type'].value_counts().to_string())

print('\nTest class distribution:')
print(df_test['type'].value_counts().to_string())


## 7 - Class Imbalance Handling: Hybrid Strategy

| Layer | Technique | Applied to |
|---|---|---|
| Data — oversample | `sklearn.resample` (with replacement) | All minority classes |
| Data — augment | WordNet synonym replacement | **Factoid only** (smallest class) |
| Loss function | Inverse-frequency `CrossEntropyLoss` weights | Every training step |

> Augmentation targets only Factoid because it is consistently the most
> underrepresented class. Applying augmentation to majority classes risks
> amplifying noise without meaningful benefit.

In [ ]:
# ── 7-A: Inspect imbalance ───────────────────────────────────────────────────
counts_train = df_train_raw['type'].value_counts()
majority_n   = int(counts_train.max())
minority_cls = counts_train.index[-1]   # auto-detect smallest class
ir           = majority_n / counts_train.min()

print(f'Majority count  : {majority_n}')
print(f'Minority class  : {minority_cls!r}  ({counts_train[minority_cls]} samples)')
print(f'Imbalance ratio : {ir:.2f}x')

In [ ]:
# ── 7-B: Synonym-replacement augmentation ───────────────────────────────────
from nltk.corpus import wordnet
import random


def _wordnet_pos(treebank_tag: str) -> str:
    '''Map Penn Treebank POS tag to WordNet POS constant.'''
    return {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV,
    }.get(treebank_tag[0], wordnet.NOUN)


def synonym_replace(sentence: str, n_swaps: int = 2) -> str:
    '''
    Replace up to n_swaps content words with a WordNet synonym.

    Medical safety rules — a token is SKIPPED if it is:
      - 3 or fewer characters  (articles, prepositions)
      - all uppercase          (medical acronyms: CBC, MRI, CT, HIV)
      - a proper noun (NNP/NNPS) — disease/drug names
      - not purely alphabetic  (preserves: HbA1c, COVID-19)

    Only single-word synonyms are accepted to avoid phrase mismatches.
    '''
    import nltk
    tokens   = nltk.word_tokenize(sentence)
    pos_tags = nltk.pos_tag(tokens)
    new_toks = tokens.copy()

    candidates = [
        i for i, (word, pos) in enumerate(pos_tags)
        if len(word) > 3
        and word.isalpha()
        and not word.isupper()             # keep acronyms
        and pos not in ('NNP', 'NNPS')    # keep proper nouns
    ]
    random.shuffle(candidates)

    replaced = 0
    for idx in candidates:
        if replaced >= n_swaps:
            break
        word, pos = pos_tags[idx]
        synsets   = wordnet.synsets(word.lower(), pos=_wordnet_pos(pos))
        synonyms  = []
        for syn in synsets:
            for lemma in syn.lemmas():
                cand = lemma.name().replace('_', ' ')
                if ' ' not in cand and cand.lower() != word.lower():
                    synonyms.append(cand)
        if synonyms:
            new_toks[idx] = random.choice(synonyms)
            replaced += 1

    return ' '.join(new_toks)


# Smoke-test
_q = 'what are the common symptoms used to diagnose appendicitis?'
print('Original :', _q)
print('Augmented:', synonym_replace(_q)) # Re-run this cell to confirm fix

In [ ]:
# ── §7-C: Build balanced training DataFrame ──────────────────────────────────

# Step 1: Oversample all minority classes to majority_n
parts = []
for cls_name in CLASS_NAMES:
    subset = df_train_raw[df_train_raw['type'] == cls_name]
    if len(subset) < majority_n:
        subset = resample(
            subset,
            replace      = True,
            n_samples    = majority_n,
            random_state = SEED
        )
    parts.append(subset)

df_oversampled = (
    pd.concat(parts)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
print(f'After oversampling : {len(df_oversampled):,} samples')

# Step 2: Synonym augmentation — Factoid only
factoid_sub = df_oversampled[df_oversampled['type'] == MINORITY_CLS].copy()
n_aug       = min(len(factoid_sub), AUG_CAP)
aug_rows    = (
    factoid_sub
    .sample(n=n_aug, replace=True, random_state=SEED)
    .reset_index(drop=True)
)
aug_rows['clean_question'] = aug_rows['clean_question'].apply(
    lambda q: synonym_replace(str(q), n_swaps=2)
)
aug_rows['label'] = aug_rows['type'].map(LABEL2ID)   # re-align after copy
print(f'Augmented {n_aug} new {MINORITY_CLS!r} samples via synonym replacement.')

# Step 3: Merge
df_train = (
    pd.concat([df_oversampled, aug_rows], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
print(f'Final balanced training set: {len(df_train):,} samples')
print(df_train['type'].value_counts().to_string())

In [ ]:
# ── §7-D: Compute class weights for CrossEntropyLoss ─────────────────────────
# Weights are computed on the ORIGINAL unbalanced distribution so they
# capture the true class skew — applied during training for extra stability.
raw_weights = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.arange(NUM_CLASSES),
    y            = df_train_raw['label'].values
)
class_weights_tensor = torch.tensor(raw_weights, dtype=torch.float32).to(device)

print('Loss function class weights:')
for i, w in enumerate(raw_weights):
    bar = '#' * int(w * 16)
    print(f'  {ID2LABEL[i]:<14}  {w:.4f}  {bar}')

In [ ]:
# ── §7-E: Before / After visualisation ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Class Imbalance Correction', fontsize=13, fontweight='bold')

for ax, df_, title in zip(
        axes,
        [df_train_raw, df_train],
        ['Before  (Original Train)', 'After  (Oversample + Factoid Aug)']):
    c    = df_['type'].value_counts()
    cols = [PALETTE.get(t, '#aaa') for t in c.index]
    bars = ax.bar(c.index, c.values, color=cols, edgecolor='black', lw=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Samples')
    ax.tick_params(axis='x', rotation=20)
    for b, v in zip(bars, c.values):
        ax.text(b.get_x()+b.get_width()/2, v+0.5, str(v),
                ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


## 8 - Tokenisation with BioBERT Tokenizer

In [ ]:
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Max length : {tokenizer.model_max_length}')
print(f'Pad token  : {tokenizer.pad_token!r}')

# Sanity-check: tokenise one example
_sample = df_train['clean_question'].iloc[0]
_enc = tokenizer(_sample, max_length=MAX_LEN, truncation=True, padding='max_length')
print(f'\nSample: {_sample[:80]}')
print(f'Token count (padded to {MAX_LEN}): {len(_enc["input_ids"])}')


## 9 - PyTorch Dataset & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader


class MedQDataset(Dataset):
    '''
    PyTorch Dataset for BioBERT sequence classification.

    Returns per item:
      - input_ids      : token IDs (LongTensor, shape MAX_LEN)
      - attention_mask : 1 for real tokens, 0 for padding (LongTensor MAX_LEN)
      - labels         : integer label 0-3 (LongTensor scalar)
    '''

    def __init__(self, df: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row['clean_question']),
            padding        = 'max_length',
            truncation     = True,
            max_length     = self.max_len,
            return_tensors = 'pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'labels'         : torch.tensor(int(row['label']), dtype=torch.long),
        }


# Instantiate datasets
train_dataset = MedQDataset(df_train, tokenizer)
val_dataset   = MedQDataset(df_val,   tokenizer)
test_dataset  = MedQDataset(df_test,  tokenizer)

print(f'Train dataset : {len(train_dataset):,} samples')
print(f'Val   dataset : {len(val_dataset):,} samples')
print(f'Test  dataset : {len(test_dataset):,} samples')

# Quick shape sanity check
_item = train_dataset[0]
for key, val in _item.items():
    print(f'  {key:<20}: {val.shape}  dtype={val.dtype}')


## 10 - Load BioBERT for Sequence Classification

In [ ]:
from transformers import AutoModelForSequenceClassification

print(f'Loading BERT: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = NUM_CLASSES,
    id2label   = ID2LABEL,
    label2id   = LABEL2ID,
)
model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total:,}')
print(f'Trainable parameters : {trainable:,}')
print(f'Device               : {next(model.parameters()).device}')


## 11 - Training with HuggingFace Trainer API

The Trainer API handles:
- Gradient accumulation and clipping
- Learning-rate scheduling (linear warmup)
- Evaluation at each epoch
- Metric computation via a custom `compute_metrics` function

> **Model selection:** best checkpoint selected by validation **Macro F1**
> (not accuracy), which is the correct metric for imbalanced datasets.

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# ── Custom metric function (called after each eval step) ─────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1);
    acc   = accuracy_score(labels, predictions)
    mf1   = f1_score(labels, predictions, average='macro',    zero_division=0)
    wf1   = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {'accuracy': acc, 'macro_f1': mf1, 'weighted_f1': wf1}


# ── Training arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = './bert_checkpoints',
    num_train_epochs             = EPOCHS,
    per_device_train_batch_size  = BATCH_SIZE,
    per_device_eval_batch_size   = BATCH_SIZE,
    learning_rate                = LR,
    weight_decay                 = WEIGHT_DECAY,
    warmup_ratio                 = WARMUP_RATIO,
    eval_strategy                = 'epoch', # Changed from evaluation_strategy
    save_strategy                = 'no',     # no checkpoint saving (per spec)
    load_best_model_at_end       = False,
    metric_for_best_model        = 'macro_f1',
    logging_strategy             = 'epoch',
    report_to                    = 'none',
    seed                         = SEED,
    fp16                         = torch.cuda.is_available(),  # mixed precision on GPU
    dataloader_num_workers       = 2,
)

print('TrainingArguments configured.')
print(f'  Epochs        : {EPOCHS}')
print(f'  Batch size    : {BATCH_SIZE}')
print(f'  LR            : {LR}')
print(f'  Mixed prec.   : {training_args.fp16}')

In [ ]:
# ── Trainer instantiation ─────────────────────────────────────────────────────
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

# ── Training ──────────────────────────────────────────────────────────────────
print('Starting BERT fine-tuning...')
print('=' * 65)
train_result = trainer.train()

print('\nTraining complete.')
print(f'  Runtime      : {train_result.metrics["train_runtime"]:.1f} s')
print(f'  Samples/sec  : {train_result.metrics["train_samples_per_second"]:.1f}')
print(f'  Final loss   : {train_result.metrics["train_loss"]:.4f}')

In [ ]:
# ── Extract per-epoch history from trainer logs ───────────────────────────────
history = {'epoch': [], 'train_loss': [], 'val_accuracy': [],
           'val_macro_f1': [], 'val_weighted_f1': []}

for log in trainer.state.log_history:
    if 'loss' in log and 'epoch' in log:
        history['epoch'].append(log['epoch'])
        history['train_loss'].append(log['loss'])
    if 'eval_accuracy' in log:
        history['val_accuracy'].append(log['eval_accuracy'])
        history['val_macro_f1'].append(log['eval_macro_f1'])
        history['val_weighted_f1'].append(log.get('eval_weighted_f1', 0.0))

print('Training log extracted.')
for e, tl, vf1 in zip(
        history['epoch'],
        history['train_loss'],
        history['val_macro_f1']):
    print(f'  Epoch {e:.0f}: train_loss={tl:.4f}  val_macro_f1={vf1:.4f}')


## 12 - Evaluation on Test Set
**All metrics computed on the held-out test split.**

In [ ]:
# ── Run inference on test set via Trainer.predict() ──────────────────────────
print('Running inference on test set...')
test_output = trainer.predict(test_dataset)

test_logits = test_output.predictions
test_preds  = np.argmax(test_logits, axis=-1)
test_labels = np.array([df_test['label'].values[i] for i in range(len(df_test))])

print(f'Test samples: {len(test_preds)}')

In [ ]:
# ── Aggregate metrics ─────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

acc      = accuracy_score(test_labels, test_preds)
macro_f1 = f1_score(test_labels, test_preds, average='macro',    zero_division=0)
wt_f1    = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
mac_prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
mac_rec  = recall_score(test_labels,   test_preds, average='macro', zero_division=0)
pc_f1    = f1_score(test_labels, test_preds, average=None, zero_division=0)

print('=' * 55)
print('  BIOBERT — TEST SET EVALUATION')
print('=' * 55)
print(f'  Accuracy          : {acc:.4f}')
print(f'  Macro Precision   : {mac_prec:.4f}')
print(f'  Macro Recall      : {mac_rec:.4f}')
print(f'  Macro F1  [KEY]   : {macro_f1:.4f}')
print(f'  Weighted F1       : {wt_f1:.4f}')
print()
print('  Per-Class F1:')
for i, f in enumerate(pc_f1):
    bar = '#' * int(f * 24)
    print(f'    {ID2LABEL[i]:<14}  {f:.4f}  {bar}')

In [ ]:
# ── Full classification report ────────────────────────────────────────────────
print('=' * 55)
print('  CLASSIFICATION REPORT')
print('=' * 55)
report = classification_report(
    test_labels, test_preds,
    target_names = CLASS_NAMES,
    zero_division = 0
)
print(report)

In [ ]:
# ── Confusion matrix (printed as DataFrame) ───────────────────────────────────
cm    = confusion_matrix(test_labels, test_preds)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)

print('CONFUSION MATRIX  (rows=True, cols=Predicted):')
print(cm_df.to_string())
print()
print('Row-normalised (Recall per class):')
print(cm_df.div(cm_df.sum(axis=1), axis=0).round(3).to_string())


## 13 - Confusion Matrix Heatmap (Seaborn)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('BioBERT — Confusion Matrix', fontsize=13, fontweight='bold')

# Left: raw counts
annot_cnt = np.array([[str(cm[i,j]) for j in range(NUM_CLASSES)]
                       for i in range(NUM_CLASSES)])
sns.heatmap(
    cm, annot=annot_cnt, fmt='s', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=axes[0], linewidths=0.5, annot_kws={'size': 12}
)
axes[0].set_title('Raw Counts', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=20)

# Right: row-normalised (= recall per class)
cm_norm   = cm.astype(float) / cm.sum(axis=1, keepdims=True)
annot_pct = np.array([[f'{cm_norm[i,j]:.0%}' for j in range(NUM_CLASSES)]
                       for i in range(NUM_CLASSES)])
sns.heatmap(
    cm_norm, annot=annot_pct, fmt='s', cmap='RdYlGn',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=axes[1], vmin=0, vmax=1,
    linewidths=0.5, annot_kws={'size': 12}
)
axes[1].set_title('Row-Normalised  (Recall per Class)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


## 14 - Training Curves

In [ ]:
if not history['epoch']:
    print('No log history available — run §11 first.')
else:
    ep_range = list(range(1, len(history['val_macro_f1']) + 1))

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle('BioBERT — Training Performance', fontsize=13, fontweight='bold')

    # (1) Training loss
    ax = axes[0]
    if history['train_loss']:
        ax.plot(ep_range, history['train_loss'][:len(ep_range)],
                'o-', color='#e74c3c', lw=2, ms=7)
        ax.fill_between(ep_range, history['train_loss'][:len(ep_range)],
                        alpha=0.12, color='#e74c3c')
        for x, y in zip(ep_range, history['train_loss']):
            ax.annotate(f'{y:.4f}', (x, y),
                        textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
    ax.set_title('Training Loss', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.grid(True, alpha=0.3)

    # (2) Validation accuracy
    ax = axes[1]
    if history['val_accuracy']:
        ax.plot(ep_range, history['val_accuracy'],
                'o-', color='#3498db', lw=2, ms=7)
        ax.fill_between(ep_range, history['val_accuracy'], alpha=0.12, color='#3498db')
        ax.set_ylim(0, 1.05)
        for x, y in zip(ep_range, history['val_accuracy']):
            ax.annotate(f'{y:.3f}', (x, y),
                        textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
    ax.set_title('Validation Accuracy', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.grid(True, alpha=0.3)

    # (3) Validation Macro F1
    ax = axes[2]
    if history['val_macro_f1']:
        ax.plot(ep_range, history['val_macro_f1'],
                'o-', color='#2ecc71', lw=2, ms=7)
        ax.fill_between(ep_range, history['val_macro_f1'], alpha=0.12, color='#2ecc71')
        ax.set_ylim(0, 1.05)
        best_ep = int(np.argmax(history['val_macro_f1'])) + 1
        ax.axvline(best_ep, color='red', ls='--', lw=1.5,
                   label=f'Best epoch {best_ep}')
        for x, y in zip(ep_range, history['val_macro_f1']):
            ax.annotate(f'{y:.3f}', (x, y),
                        textcoords='offset points', xytext=(0, 9), ha='center', fontsize=9)
        ax.legend(fontsize=9)
    ax.set_title('Validation Macro F1', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Macro F1'); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 15 - Sample Predictions (Correct & Incorrect)

In [ ]:
import torch.nn.functional as F

# Compute per-sample probabilities from logits
all_probs = F.softmax(
    torch.tensor(test_logits, dtype=torch.float32), dim=1
).numpy()  # shape (n_test, 4)


def show_predictions(df_test, preds, labels, probs,
                     n_correct=5, n_wrong=5):
    correct_idx = [i for i in range(len(preds)) if preds[i] == labels[i]]
    wrong_idx   = [i for i in range(len(preds)) if preds[i] != labels[i]]

    print('=' * 76)
    print(f'  CORRECT PREDICTIONS  [{len(correct_idx)}/{len(preds)} correct]')
    print('=' * 76)
    for i in correct_idx[:n_correct]:
        row  = df_test.iloc[i]
        conf = probs[i].max()
        prob_str = {ID2LABEL[j]: f'{probs[i][j]:.2f}' for j in range(NUM_CLASSES)}
        print(f'\n  [CORRECT]  Confidence: {conf:.1%}')
        print(f'  Question  : {str(row["question"])[:115]}')
        print(f'  True      : {row["type"]}')
        print(f'  Predicted : {ID2LABEL[preds[i]]}')
        print(f'  Probs     : {prob_str}')

    print('\n' + '=' * 76)
    print(f'  INCORRECT PREDICTIONS  [{len(wrong_idx)}/{len(preds)} wrong]')
    print('=' * 76)
    for i in wrong_idx[:n_wrong]:
        row     = df_test.iloc[i]
        conf    = probs[i].max()
        true_cf = probs[i][labels[i]]
        prob_str = {ID2LABEL[j]: f'{probs[i][j]:.2f}' for j in range(NUM_CLASSES)}
        print(f'\n  [WRONG]  Pred-conf: {conf:.1%}  |  True-class-conf: {true_cf:.1%}')
        print(f'  Question  : {str(row["question"])[:115]}')
        print(f'  True      : {row["type"]}')
        print(f'  Predicted : {ID2LABEL[preds[i]]}')
        print(f'  Probs     : {prob_str}')

    print('\n' + '=' * 76)
    print('  ERROR ANALYSIS — Confused Pairs (True -> Predicted)')
    print('=' * 76)
    pairs = {}
    for i in wrong_idx:
        p = f'{ID2LABEL[labels[i]]}  ->  {ID2LABEL[preds[i]]}'
        pairs[p] = pairs.get(p, 0) + 1
    for pair, cnt in sorted(pairs.items(), key=lambda x: -x[1]):
        print(f'  {pair:<40}  {cnt} errors  {"#" * cnt}')


show_predictions(df_test, test_preds, test_labels, all_probs,
                 n_correct=5, n_wrong=5)


## 16 - Prediction Function
**Ready-to-use single-question inference.**

In [ ]:
def predict_question(text: str) -> dict:
    '''
    Classify a single medical question using the fine-tuned BioBERT model.

    Args:
        text (str): Raw medical question text.

    Returns:
        dict with keys:
          - prediction  : str  ('Diagnostic' / 'Factoid' / 'Preventive' / 'Prognostic')
          - confidence  : float (0-1, probability of predicted class)
          - all_probs   : dict  {class: probability}
    '''
    model.eval()

    # Preprocess exactly as during training
    clean = preprocess_text(text)

    enc = tokenizer(
        clean,
        padding        = 'max_length',
        truncation     = True,
        max_length     = MAX_LEN,
        return_tensors = 'pt',
    )
    input_ids = enc['input_ids'].to(device)
    attn_mask = enc['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

    pred_id   = int(probs.argmax())
    pred_label = ID2LABEL[pred_id]
    confidence = float(probs[pred_id])

    return {
        'prediction' : pred_label,
        'confidence' : confidence,
        'all_probs'  : {ID2LABEL[i]: float(p) for i, p in enumerate(probs)},
    }


# ── Demonstration ─────────────────────────────────────────────────────────────
demo_questions = [
    'What bacteria most commonly causes community-acquired pneumonia?',
    'What is the diagnostic criteria for type 2 diabetes mellitus?',
    'Which vaccine is recommended to prevent hepatitis B in newborns?',
    'What is the 5-year survival rate for stage III colorectal cancer?',
]

print('=' * 72)
print('  PREDICTION FUNCTION DEMO')
print('=' * 72)
for q in demo_questions:
    result = predict_question(q)
    print(f'\nQ: {q}')
    print(f'   Prediction : {result["prediction"]}  (confidence={result["confidence"]:.1%})')
    prob_str = {k: f'{v:.3f}' for k, v in result['all_probs'].items()}
    print(f'   All probs  : {prob_str}')


## 17 - Additional Visualisations

In [ ]:
# ── Per-class F1 bar chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('BERT — Detailed Performance', fontsize=13, fontweight='bold')

# (1) Per-class F1
colors = [PALETTE[n] for n in CLASS_NAMES]
bars   = axes[0].bar(CLASS_NAMES, pc_f1, color=colors,
                     edgecolor='black', linewidth=0.5, width=0.55)
axes[0].axhline(macro_f1, color='#2c3e50', ls='--', lw=1.8,
                label=f'Macro F1 = {macro_f1:.3f}')
axes[0].set_title('Per-Class F1 Score', fontweight='bold')
axes[0].set_ylabel('F1 Score'); axes[0].set_ylim(0, 1.15)
axes[0].tick_params(axis='x', rotation=20); axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
for b, v in zip(bars, pc_f1):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.025,
                 f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

# (2) Prediction distribution vs true distribution
x   = np.arange(NUM_CLASSES); w = 0.35
true_counts = pd.Series(test_labels).value_counts().sort_index()
pred_counts = pd.Series(test_preds).value_counts().sort_index()
b1 = axes[1].bar(x - w/2, true_counts.values, w, label='True',
                 color='#3498db', edgecolor='black', lw=0.5)
b2 = axes[1].bar(x + w/2, pred_counts.values, w, label='Predicted',
                 color='#e74c3c', edgecolor='black', lw=0.5)
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES, rotation=20)
axes[1].set_title('True vs Predicted Distribution', fontweight='bold')
axes[1].set_ylabel('Count'); axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
model.save_pretrained("bert_model")
tokenizer.save_pretrained("bert_model")